# Myst Architecture: Timing Analysis & Comparison
## Classical (ECC) vs. Post-Quantum (Kyber)

This notebook simulates the entire lifecycle of the distributed trust architecture. It measures the execution time for the three critical stages:

1.  **Distributed Key Generation (DKPG)**: Establishing the root of trust.
2.  **Encryption**: Securing the file/key (Host -> Quorum).
3.  **Decryption**: Recovering the file/key (Quorum -> Host).

In [ ]:
import time
import os
import pandas as pd
from myst_simulation.ic import MystProcessingIC
from myst_simulation.host import Host
from myst_simulation.protocols import (
    run_ecc_dkpg, run_ecc_decryption_request,
    run_pqc_dkpg, run_pqc_distribute_secret
)

# Configuration
QUORUM_SIZE = 5
FILE_SIZE_MB = 10
dummy_data = os.urandom(FILE_SIZE_MB * 1024 * 1024)
results = []

## 1. Classical Simulation (ECC - ElGamal)

In [ ]:
print("--- Running Classical (ECC) Simulation ---")

# Setup
ecc_quorum = [MystProcessingIC(f"IC_{i}") for i in range(QUORUM_SIZE)]
ecc_host = Host("Alice")

# --- TIMING DKPG ---
start = time.perf_counter()
agg_key = run_ecc_dkpg(ecc_quorum)
ecc_host.Y_agg = agg_key
ecc_dkpg_time = (time.perf_counter() - start) * 1000 # ms
print(f"ECC DKPG Time: {ecc_dkpg_time:.2f} ms")

# --- TIMING ENCRYPTION ---
# Note: In Classical, Encryption is mostly local on the Host (ElGamal).
start = time.perf_counter()
c1, c2 = ecc_host.encrypt_ecc(dummy_data)
ecc_enc_time = (time.perf_counter() - start) * 1000
print(f"ECC Encryption Time: {ecc_enc_time:.2f} ms")

# --- TIMING DECRYPTION ---
# Host requests shares from all ICs + Aggregates them
start = time.perf_counter()
shares = run_ecc_decryption_request(ecc_host, c1, ecc_quorum)
ecc_host.decrypt_ecc(c2, shares)
ecc_dec_time = (time.perf_counter() - start) * 1000
print(f"ECC Decryption Time: {ecc_dec_time:.2f} ms")

results.append({
    "Protocol": "Classical (ECC)",
    "KeyGen (ms)": ecc_dkpg_time,
    "Encryption (ms)": ecc_enc_time,
    "Decryption (ms)": ecc_dec_time
})

## 2. Post-Quantum Simulation (Kyber + AES)

In [ ]:
print("\n--- Running PQC (Kyber) Simulation ---")

# Setup
pqc_quorum = [MystProcessingIC(f"IC_{i}") for i in range(QUORUM_SIZE)]
pqc_host = Host("Bob")
file_key = os.urandom(32) # AES-256 Key

# --- TIMING DKPG ---
# In PQC, this is generating Kyber keys and sending PubKeys to Host
start = time.perf_counter()
all_keys = run_pqc_dkpg(pqc_quorum)
pqc_host.set_quorum_pqc_keys(all_keys)
pqc_dkpg_time = (time.perf_counter() - start) * 1000
print(f"PQC KeyGen Time: {pqc_dkpg_time:.2f} ms")

# --- TIMING ENCRYPTION ---
# In PQC, 'Encryption' involves Splitting + Kyber Encap + Network Transmission
start = time.perf_counter()
pqc_host.encrypt_pqc(file_key, pqc_quorum)
pqc_enc_time = (time.perf_counter() - start) * 1000
print(f"PQC Encryption Time: {pqc_enc_time:.2f} ms")

# --- TIMING DECRYPTION ---
# Quorum sends shares back -> Host Reconstructs
start = time.perf_counter()
# 1. Fetch shares (Simulated network call)
received_shares = [ic.pqc_retrieve_share_for_decryption(None) for ic in pqc_quorum]
# 2. Reconstruct
recovered_key = pqc_host.decrypt_pqc(received_shares)
pqc_dec_time = (time.perf_counter() - start) * 1000
print(f"PQC Decryption Time: {pqc_dec_time:.2f} ms")

results.append({
    "Protocol": "Post-Quantum (Kyber)",
    "KeyGen (ms)": pqc_dkpg_time,
    "Encryption (ms)": pqc_enc_time,
    "Decryption (ms)": pqc_dec_time
})

## 3. Comparison Results

In [ ]:
df = pd.DataFrame(results)
print("\n--- Final Timing Comparison ---")
print(df)

# Analysis of results:
# 1. KeyGen: PQC should be significantly faster (Linear Scaling vs Quadratic).
# 2. Encryption: PQC will be slower because it involves N Kyber Encapsulations + Network calls,
#    whereas Classical ECC encryption is just a local math operation.
# 3. Decryption: Both scale linearly, but PQC involves Decapsulation which might be heavier/lighter depending on implementation.